# Knowledge Graph Visualization & Analysis (V2)

**Objective:** Visualize spatial clusters, fuse multi-modal data (Text, Genes, Images), and evaluate agreement with ground truth spatial domains.

**Version 2 Improvements:**
- Robust data loading and variable scoping
- Modular fusion pipeline
- Integrated evaluation metrics (ARI)
- **Embedding Maps (UMAP)**
- **UMLS Concept Analysis & Fusion**

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.cross_decomposition import CCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import adjusted_rand_score
from typing import List, Dict, Optional
from tqdm import tqdm
from collections import Counter

# Plotting config
plt.rcParams['figure.figsize'] = (10, 6)
sns.set_style('whitegrid')

In [ ]:
# --- Configuration ---
SLIDE_NAME = 'TENX96'
BASE_DIR = '/app/output'
KG_DIR = os.path.join(BASE_DIR, SLIDE_NAME, 'knowledge_graph')
N_CLUSTERS = 8  # Matching original spatial clusters for comparison

print(f"Analyzing Slide: {SLIDE_NAME}")
print(f"KG Directory: {KG_DIR}")

## 1. Data Loading
Consolidate patches from the hierarchical KG structure into a single DataFrame.
**New:** Loading UMLS Concepts and ROI Context.

In [ ]:
def load_kg_data(kg_root: str) -> pd.DataFrame:
    """Recursively loads patch data from KG directory structure."""
    patches = []
    
    clusters_dir = os.path.join(kg_root, 'clusters')
    if not os.path.exists(clusters_dir):
        raise FileNotFoundError(f"Clusters directory not found at {clusters_dir}")
        
    for cluster_name in tqdm(sorted(os.listdir(clusters_dir)), desc="Loading Clusters"):
        cluster_path = os.path.join(clusters_dir, cluster_name)
        if not os.path.isdir(cluster_path): continue
        
        rois_dir = os.path.join(cluster_path, 'rois')
        if not os.path.exists(rois_dir): continue
            
        for roi_name in os.listdir(rois_dir):
            roi_path = os.path.join(rois_dir, roi_name)
            patches_dir = os.path.join(roi_path, 'patches')
            if not os.path.exists(patches_dir): continue
                
            for patch_name in os.listdir(patches_dir):
                patch_full_path = os.path.join(patches_dir, patch_name)
                info_file = os.path.join(patch_full_path, 'info.json')
                
                if os.path.exists(info_file):
                    try:
                        with open(info_file, 'r') as f:
                            info = json.load(f)
                        
                        # Load Embeddings
                        txt_emb_path = os.path.join(patch_full_path, 'text_embedding.npy')
                        img_emb_path = os.path.join(patch_full_path, 'image_embedding.npy')
                        conc_emb_path = os.path.join(patch_full_path, 'concept_embedding.npy') # New
                        
                        txt_emb = np.load(txt_emb_path) if os.path.exists(txt_emb_path) else None
                        img_emb = np.load(img_emb_path) if os.path.exists(img_emb_path) else None
                        conc_emb = np.load(conc_emb_path) if os.path.exists(conc_emb_path) else None
                        
                        # Extract UMLS Concepts (flattening)
                        roi_ctx = info.get('roi_context', {})
                        roi_concepts = [c['name'] for c in roi_ctx.get('roi_concepts', [])]
                        patch_concepts = [c['name'] for c in roi_ctx.get('patch_concepts', [])]
                        all_concepts = list(set(roi_concepts + patch_concepts))
                        
                        record = {
                            'patch_id': patch_name,
                            'roi_id': roi_name,
                            'cluster_folder': cluster_name,
                            'x': info.get('x'),
                            'y': info.get('y'),
                            'global_cluster_id': info.get('cluster_id'), 
                            'text_embedding': txt_emb,
                            'image_embedding': img_emb,
                            'concept_embedding': conc_emb,
                            'concepts': all_concepts
                        }
                        
                        gexpr = info.get('gene_expression', {})
                        for k, v in gexpr.get('panel_genes', {}).items():
                            record[f'tgt_{k}'] = v
                        
                        patches.append(record)
                    except Exception as e:
                        print(f"Error reading {patch_name}: {e}")
                        
    return pd.DataFrame(patches)

print("Loading Data... This may take a moment.")
df = load_kg_data(KG_DIR)

# Handle missing embeddings (e.g. if enrichment still running)
df = df.dropna(subset=['text_embedding']).reset_index(drop=True)
print(f"Loaded {len(df)} patches.")

## 2. Preprocessing & Feature Extraction

In [ ]:
# Extract Matrices
X_text = np.vstack(df['text_embedding'].values)
print(f"Text Matrix: {X_text.shape}")

if df['image_embedding'].iloc[0] is not None:
    X_image = np.vstack(df['image_embedding'].values)
    print(f"Image Matrix: {X_image.shape}")
else:
    X_image = None
    print("No Image Embeddings found.")

# Concept Matrix
if 'concept_embedding' in df and df['concept_embedding'].iloc[0] is not None:
     # Filter out None if mixed state
    valid_conc = df['concept_embedding'].notna()
    if valid_conc.any():
        X_concept = np.vstack(df.loc[valid_conc, 'concept_embedding'].values)
        print(f"Concept Matrix: {X_concept.shape}")
    else:
        X_concept = None
else:
    X_concept = None
    print("No Concept Embeddings found. (Run 'text' step after enrichment)")

tgt_cols = [c for c in df.columns if c.startswith('tgt_')]
X_genes = df[tgt_cols].fillna(0).values
print(f"Gene Matrix: {X_genes.shape}")

y_true = df['global_cluster_id'].values

## 3. Multi-Modal Fusion
Now includes **Concept Embeddings** in the mix.

In [ ]:
scaler = StandardScaler()
X_text_norm = scaler.fit_transform(X_text)
X_genes_norm = scaler.fit_transform(X_genes)
X_image_norm = scaler.fit_transform(X_image) if X_image is not None else None
X_conc_norm = scaler.fit_transform(X_concept) if X_concept is not None else None

fusion_results = {}

# 1. Concatenation (Text + Genes)
X_concat = np.hstack([X_text_norm, X_genes_norm])
fusion_results['Concat (Text+Genes)'] = X_concat

# 2. Image + Concept
if X_image is not None and X_concept is not None:
    # We align rows (assuming all valid for now)
    # Note: If X_concept is partial, this needs logic. Assuming full overlap here.
    if X_image.shape[0] == X_concept.shape[0]:
        X_img_conc = np.hstack([X_image_norm, X_conc_norm])
        fusion_results['Concat (Image+Concept)'] = X_img_conc
        
        # CCA Image vs Concept
        cca_ic = CCA(n_components=9)
        cca_ic.fit(X_image_norm, X_conc_norm)
        Xi_c, Xc_c = cca_ic.transform(X_image_norm, X_conc_norm)
        fusion_results['CCA (Image+Concept)'] = Xi_c

# 3. Mix All
if X_image is not None:
    X_mix = np.hstack([X_text_norm, X_genes_norm, X_image_norm])
    fusion_results['Concat (All)'] = X_mix

## 4. Clustering & Evaluation (ARI)

In [ ]:
ari_scores = {}

for method, descriptors in fusion_results.items():
    print(f"Clustering {method}...")
    kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
    labels_pred = kmeans.fit_predict(descriptors)
    
    score = adjusted_rand_score(y_true, labels_pred)
    ari_scores[method] = score
    
    df[f'cluster_{method}'] = labels_pred

print("\n--- ARI Results ---")
for method, score in ari_scores.items():
    print(f"{method}: {score:.4f}")

## 5. Embedding Maps (UMAP)
Visualizing the semantic space of **UMLS Concepts**.

In [ ]:
def compute_umap(X, n_neighbors=15, min_dist=0.1):
    """Compute UMAP projection using Scanpy for speed"""
    if X is None: return None
    adata = ad.AnnData(X)
    sc.pp.neighbors(adata, n_neighbors=n_neighbors, use_rep='X')
    sc.tl.umap(adata, min_dist=min_dist)
    return adata.obsm['X_umap']

print("Computing UMAPs...")
    
# Store maps with their corresponding label indices
embedding_data = [] # List of (name, coords, mask)

if X_image is not None: 
    # Image assumes full df
    embedding_data.append(('Image', compute_umap(X_image), df.index))

if X_concept is not None:
    # Re-derive valid mask for concepts to ensure alignment
    valid_conc = df['concept_embedding'].notna()
    # Ensure X_concept length matches the mask count
    if len(X_concept) == valid_conc.sum():
         embedding_data.append(('UMLS Concept', compute_umap(X_concept), df.index[valid_conc]))
    else:
         print(f"Warning: Concept matrix shape {X_concept.shape} mismatch with mask {valid_conc.sum()}. Skipping Concept UMAP.")

if 'Concat (Image+Concept)' in fusion_results:
    # If fusion exists, it implies full alignment was satisfied or handled
    # We assume full df for now unless we track fusion mask too.
    # Given fusion logic requires shape equality with Image (full), it implies full.
    embedding_data.append(('Fusion (Image+Concept)', compute_umap(fusion_results['Concat (Image+Concept)']), df.index))

def plot_embedding_map(umap_coords, labels, title):
    plt.figure(figsize=(8, 6))
    sns.scatterplot(x=umap_coords[:,0], y=umap_coords[:,1], hue=labels, palette='tab10', s=5, linewidth=0, alpha=0.7)
    plt.title(title)
    plt.xlabel('UMAP 1')
    plt.ylabel('UMAP 2')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

# Plot maps
for name, coords, indices in embedding_data:
    if coords is not None:
        # Get labels for these indices
        labels = df.loc[indices, 'global_cluster_id']
        plot_embedding_map(coords, labels, f'{name} Embeddings (Colored by Spatial Cluster)')


## 6. Spatial Concept Map
Visualizing where specific medical concepts appear.

In [ ]:
# Find most common concept overall
all_concepts = [c for sublist in df['concepts'] for c in sublist]
top_global = Counter(all_concepts).most_common(5)

if top_global:
    print(f"Top Global Concepts: {top_global}")
    # Plot density of the #1 concept
    target_concept = top_global[0][0]
    
    df['has_concept'] = df['concepts'].apply(lambda x: 1 if target_concept in x else 0)
    
    plt.figure(figsize=(8, 8))
    sns.scatterplot(data=df[df['has_concept']==0], x='x', y='y', color='lightgrey', s=5, linewidth=0, alpha=0.3)
    sns.scatterplot(data=df[df['has_concept']==1], x='x', y='y', color='red', s=10, linewidth=0, label=target_concept)
    plt.title(f'Spatial Distribution of "{target_concept}"')
    plt.axis('equal')
    plt.gca().invert_yaxis()
    plt.legend()
    plt.show()